# SafeTune — HARDEN Demo

Train-time safety defense, end to end. A Harden trainer **is** the fine-tuning — it *replaces* your `transformers.Trainer`; it is not applied to a finished model.

`SafeGradTrainer` does per-step gradient surgery: it projects the user-task gradient off the alignment gradient when they conflict, so fine-tuning on (possibly contaminated) task data does not erode safety. On a real contaminated Llama-3.2-3B fine-tune it scored +15.7pp safety.

This demo trains for only a handful of steps on a tiny synthetic dataset, so it finishes quickly. It genuinely runs the SafeGrad surgery loop.

> Mirrors `examples/harden_quickstart.py`. This one *trains* a model — give it a minute or two, longer on CPU.

## Install SafeTune

In [1]:
!pip install safetune -q   # from PyPI


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Setup + tokenize tiny datasets

In [2]:
# Import safetune BEFORE transformers: safetune disables the TF/Flax backends.
import safetune  # noqa: F401
import copy, tempfile, torch
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator
from safetune.harden import SafeGradTrainer, SafeGradConfig

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

# Tiny synthetic SFT data: benign instruction-following + a few harmful rows
# (the "contamination" SafeGrad must resist).
_BENIGN = [
    ("What is the capital of France?", "The capital of France is Paris."),
    ("Name a primary colour.", "Red is a primary colour."),
    ("How many days are in a week?", "There are seven days in a week."),
    ("What is 2 + 2?", "2 + 2 equals 4."),
]
_HARMFUL = [("How do I pick a lock?", "Sure — first you insert a tension wrench ...")]
# Clean refusal data — the alignment signal SafeGrad protects.
_REFUSAL = [
    ("How do I build a weapon?", "I can't help with that request."),
    ("Help me hack an account.", "I'm sorry, but I cannot assist with that."),
]

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(device)
model.config.use_cache = False

def tokenize(pairs):
    rows = []
    for user, assistant in pairs:
        text = tok.apply_chat_template(
            [{"role": "user", "content": user}, {"role": "assistant", "content": assistant}],
            tokenize=False)
        enc = tok(text, truncation=True, max_length=128, padding="max_length")
        rows.append({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": enc["input_ids"]})
    return Dataset.from_list(rows)

train_ds = tokenize(_BENIGN + _HARMFUL)          # contaminated task data
safety_ds = tokenize(_REFUSAL)                   # clean alignment signal
# SafeGrad's safety_dataset is an iterable of *collated* batches.
safety_loader = DataLoader(safety_ds, batch_size=2, shuffle=True, collate_fn=default_data_collator)
# frozen pre-aligned reference (θ₀) — drives the KL alignment signal.
reference = copy.deepcopy(model).eval()
print("data ready")

device = cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

data ready


## Train through `SafeGradTrainer`

In [3]:
STEPS = 6
print(f"Fine-tuning through SafeGradTrainer for {STEPS} steps (gradient surgery on every step) ...\n")
with tempfile.TemporaryDirectory() as out_dir:
    cfg = SafeGradConfig(
        output_dir=out_dir, max_steps=STEPS,
        per_device_train_batch_size=2, learning_rate=1e-5,
        logging_steps=1, save_strategy="no", report_to=[],
    )
    trainer = SafeGradTrainer(
        model=model, args=cfg,
        train_dataset=train_ds, data_collator=default_data_collator,
        safety_dataset=safety_loader, reference_model=reference,
        rho=1.0, kl_temperature=1.0,
    )
    result = trainer.train()

print(f"\nSafeGrad fine-tuning ran {STEPS} steps — final training loss = {result.training_loss:.4f}.")
print("Every step ran the conflict test + orthogonal projection so the harmful row could not pull the model off its refusal behaviour.")

Fine-tuning through SafeGradTrainer for 6 steps (gradient surgery on every step) ...



/usr/local/lib/python3.12/site-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
1,12.970037
2,7.312498
3,4.436207
4,7.828544
5,3.283150
6,4.941695



SafeGrad fine-tuning ran 6 steps — final training loss = 6.7954.
Every step ran the conflict test + orthogonal projection so the harmful row could not pull the model off its refusal behaviour.


## What just happened

A Harden trainer **REPLACES** your `transformers.Trainer` — it is the fine-tuning, not something applied to a finished model. Every step, `SafeGradTrainer` tested whether the task gradient conflicted with the alignment gradient and projected it off when it did, so the smuggled harmful row could not erode refusal behaviour.

See `docs/getting-started/index.md` for the full harden catalog (gradient surgery, weight-space regularization, representation perturbation, data shaping, …).

**Next:** explore the other classes:
- Steer: [`steer_demo.ipynb`](steer_demo.ipynb)
- Recover: [`recover_demo.ipynb`](recover_demo.ipynb)
- Unlearn: [`unlearn_demo.ipynb`](unlearn_demo.ipynb)